# Atlas synthetic recovery: two-process protocol

This notebook has a **fail-closed opt-in**: without `RDF_RUN_ATLAS_RECOVERY=1`, it records `NOT_TESTED` and does not touch the hosted database. To execute an authorized live test, configure the owner-only `.env` and install `pymongo`, then run the notebook in the checked-out repo.

In [ ]:
import os, json, subprocess, sys, uuid
from pathlib import Path
ROOT=Path.cwd()
assert (ROOT/'scripts/atlas_smoke.py').is_file()
LIVE=os.getenv('RDF_RUN_ATLAS_RECOVERY')=='1'
print('HOSTED_ATLAS', 'ENABLED' if LIVE else 'NOT_TESTED')

In [ ]:
if LIVE:
    run_id='RDF-SYNTH-'+uuid.uuid4().hex[:20]
    receipts=[]
    for op in ('write','read'):
        p=subprocess.run([sys.executable,str(ROOT/'scripts/atlas_smoke.py'),op,run_id],cwd=ROOT,capture_output=True,text=True,timeout=45)
        assert p.returncode==0, f'Atlas {op} failed (exit code {p.returncode}); stderr withheld for credential safety'
        rec=json.loads(p.stdout)
        assert rec['status']=='PASS_ATLAS_READBACK'
        receipts.append(rec)
    assert receipts[0]['readback_sha256']==receipts[1]['readback_sha256']
    assert receipts[1]['recovery']=='PASS_NEW_PROCESS'
    print(json.dumps({'outcome':'PASS_TWO_PROCESS_READBACK','synthetic_id':receipts[1]['ticket_id'],'digest':receipts[1]['readback_sha256']},indent=2))
else:
    print('NOT_TESTED: no hosted access attempted. Operator must explicitly enable RDF_RUN_ATLAS_RECOVERY=1.')

## Claim ceiling

An enabled run proves a synthetic write/readback across two Python processes. It does not prove kill/restart of a long-lived agent worker, distributed durability, real participant accessibility, or cryptographic signing. Keep execution outputs and provenance with the review artifacts.